In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D


## You must generate the embedding csvs before this notebook can run


# PCA

In [ ]:
before_df = pd.read_csv("embeddings_before.csv")
after_df  = pd.read_csv("embeddings_after.csv")
print(before_df.shape, after_df.shape)
before_df.head()


before_df = pd.read_csv("embeddings_before.csv")
after_df  = pd.read_csv("embeddings_after.csv")
before_df = before_df[before_df["label"]==1.0]
after_df  = after_df[after_df["label"]==1.0]


In [ ]:

fraud_cols = [c for c in before_df.columns if c.startswith("fraud_emb_")]
real_cols  = [c for c in before_df.columns if c.startswith("real_emb_")]

fraud_before = before_df[fraud_cols].values
real_before  = before_df[real_cols].values

fraud_after = after_df[fraud_cols].values
real_after  = after_df[real_cols].values

print(fraud_before.shape, fraud_after.shape)


In [ ]:
pca = PCA(n_components=3, random_state=42)

all_embeddings = np.vstack([
    fraud_before,
    real_before,
    fraud_after,
    real_after
])

pca.fit(all_embeddings)

fraud_before_pca = pca.transform(fraud_before)
real_before_pca  = pca.transform(real_before)

fraud_after_pca = pca.transform(fraud_after)
real_after_pca  = pca.transform(real_after)


In [ ]:
final_df = before_df.copy()

sample_idx = final_df.sample(n=5, random_state=3).index

fraud_pca_sample = fraud_after_pca[sample_idx]
real_pca_sample  = real_after_pca[sample_idx]

fraud_words = final_df.loc[sample_idx, "fraudulent_name"].values
real_words  = final_df.loc[sample_idx, "real_name"].values

colors = plt.cm.tab10(np.linspace(0, 1, len(sample_idx)))


In [ ]:

sample_idx = after_df.sample(n=5, random_state=3).index
fraud_words = after_df.loc[sample_idx, "fraudulent_name"].values
real_words  = after_df.loc[sample_idx, "real_name"].values
colors = plt.cm.tab10(np.linspace(0, 1, len(sample_idx)))

fraud_before_s = fraud_after_pca[sample_idx]
real_before_s  = real_after_pca[sample_idx]

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

for i, color in enumerate(colors):
    ax.scatter(*fraud_before_s[i], color=color, s=60, alpha=0.85)
    ax.scatter(*real_before_s[i],  color=color, s=60, alpha=0.85, marker="^")

    ax.text(*fraud_before_s[i], fraud_words[i], fontsize=9)
    ax.text(*real_before_s[i],  real_words[i],  fontsize=9)

ax.set_title("Embedding Space Before VATE")
x_lims = ax.get_xlim()
y_lims = ax.get_ylim()
z_lims = ax.get_zlim()

x_ticks = ax.get_xticks()
y_ticks = ax.get_yticks()
z_ticks = ax.get_zticks()

plt.show()


In [ ]:

before_df = pd.read_csv("embeddings_before.csv")
after_df  = pd.read_csv("embeddings_after.csv")

fraud_emb_cols = [c for c in before_df.columns if c.startswith("fraud_emb_")]
real_emb_cols  = [c for c in before_df.columns if c.startswith("real_emb_")]

fraud_before = before_df[fraud_emb_cols].to_numpy()
real_before  = before_df[real_emb_cols].to_numpy()

fraud_after  = after_df[fraud_emb_cols].to_numpy()
real_after   = after_df[real_emb_cols].to_numpy()

all_embeddings = np.vstack([
    fraud_before, real_before,
    fraud_after,  real_after
])

pca = PCA(n_components=3, random_state=0)
all_pca = pca.fit_transform(all_embeddings)

N = len(before_df)

fraud_before_pca = all_pca[0:N]
real_before_pca  = all_pca[N:2*N]
fraud_after_pca  = all_pca[2*N:3*N]
real_after_pca   = all_pca[3*N:4*N]


sample_idx = before_df.sample(n=2, random_state=0).index.to_numpy()

fraud_names = before_df.loc[sample_idx, "fraudulent_name"].values
real_names  = before_df.loc[sample_idx, "real_name"].values

colors = plt.cm.tab10(np.linspace(0, 1, len(sample_idx)))


fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

for i, color in enumerate(colors):
    b_f = fraud_before_pca[sample_idx[i]]
    b_r = real_before_pca[sample_idx[i]]
    a_f = fraud_after_pca[sample_idx[i]]
    a_r = real_after_pca[sample_idx[i]]

    ax.scatter(*b_f, color="blue", s=60, alpha=0.3, marker="o")
    ax.scatter(*b_r, color="blue", s=60, alpha=0.3, marker="o")

    ax.scatter(*a_f, color="orange", s=90, alpha=0.8, marker="o")
    ax.scatter(*a_r, color="orange", s=90, alpha=0.8, marker="o")

    ax.plot([b_f[0], b_r[0]], [b_f[1], b_r[1]], [b_f[2], b_r[2]],
            color="blue", linestyle="--", alpha=0.6)

    ax.plot([a_f[0], a_r[0]], [a_f[1], a_r[1]], [a_f[2], a_r[2]],
            color="orange", linewidth=2)
    
    ax.text(*b_f, fraud_names[i] + " (B)", fontsize=8)
    ax.text(*b_r, real_names[i] + " (B)", fontsize=8)
    ax.text(*a_f, fraud_names[i] + " (A)", fontsize=9)
    ax.text(*a_r, real_names[i] + " (A)", fontsize=9)

ax.set_title("Embedding Space Before and After VA-TE")
ax.grid(False)
plt.show()


In [ ]:

before_df = pd.read_csv("embeddings_before.csv")
after_df  = pd.read_csv("embeddings_after.csv")

fraud_cols = [c for c in before_df.columns if c.startswith("fraud_emb_")]
real_cols  = [c for c in before_df.columns if c.startswith("real_emb_")]

fraud_before = before_df[fraud_cols].to_numpy()
real_before  = before_df[real_cols].to_numpy()
fraud_after  = after_df[fraud_cols].to_numpy()
real_after   = after_df[real_cols].to_numpy()


dist_before = np.linalg.norm(fraud_before - real_before, axis=1)
dist_after  = np.linalg.norm(fraud_after  - real_after,  axis=1)

#some stats
print("BEFORE distances:")
print("  mean   =", dist_before.mean())
print("  median =", np.median(dist_before))
print("  std    =", dist_before.std())

print("\nAFTER distances:")
print("  mean   =", dist_after.mean())
print("  median =", np.median(dist_after))
print("  std    =", dist_after.std())

print("\nChange (AFTER - BEFORE):")
print("  mean change   =", (dist_after - dist_before).mean())
print("  median change =", np.median(dist_after - dist_before))


In [ ]:

before_df = pd.read_csv("embeddings_before.csv")
after_df  = pd.read_csv("embeddings_after.csv")

fraud_emb_cols = [c for c in before_df.columns if c.startswith("fraud_emb_")]
real_emb_cols  = [c for c in before_df.columns if c.startswith("real_emb_")]

fraud_before = before_df[fraud_emb_cols].to_numpy()
real_before  = before_df[real_emb_cols].to_numpy()

fraud_after  = after_df[fraud_emb_cols].to_numpy()
real_after   = after_df[real_emb_cols].to_numpy()

dist_before = np.linalg.norm(fraud_before - real_before, axis=1)
dist_after  = np.linalg.norm(fraud_after  - real_after,  axis=1)
change = dist_after - dist_before

mean_change = change.mean()


window = 0.05 * abs(mean_change)  # 5% window around mean
candidate_idx = np.where(np.abs(change - mean_change) <= window)[0]

if len(candidate_idx) == 0:
    candidate_idx = np.arange(len(change))

sample_idx = np.random.choice(candidate_idx, size=1, replace=False)

all_embeddings = np.vstack([
    fraud_before, real_before,
    fraud_after,  real_after
])

pca = PCA(n_components=3, random_state=0)
all_pca = pca.fit_transform(all_embeddings)

N = len(before_df)

fraud_before_pca = all_pca[0:N]
real_before_pca  = all_pca[N:2*N]
fraud_after_pca  = all_pca[2*N:3*N]
real_after_pca   = all_pca[3*N:4*N]

fraud_names = before_df.loc[sample_idx, "fraudulent_name"].values
real_names  = before_df.loc[sample_idx, "real_name"].values
colors = plt.cm.tab10(np.linspace(0, 1, len(sample_idx)))

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

for i, color in enumerate(colors):
    b_f = fraud_before_pca[sample_idx[i]]
    b_r = real_before_pca[sample_idx[i]]
    a_f = fraud_after_pca[sample_idx[i]]
    a_r = real_after_pca[sample_idx[i]]

    ax.scatter(*b_f, color=color, s=60, alpha=0.5, marker="o")
    ax.scatter(*b_r, color=color, s=60, alpha=0.5, marker="o")

    ax.scatter(*a_f, color=color, s=90, alpha=0.9, marker="^")
    ax.scatter(*a_r, color=color, s=90, alpha=0.9, marker="^")

    ax.plot([b_f[0], b_r[0]], [b_f[1], b_r[1]], [b_f[2], b_r[2]],
            color=color, linestyle="--", alpha=0.6)
    ax.plot([a_f[0], a_r[0]], [a_f[1], a_r[1]], [a_f[2], a_r[2]],
            color=color, linewidth=2)

    ax.text(*b_f, fraud_names[i] + " (B)", fontsize=8)
    ax.text(*b_r, real_names[i] + " (B)", fontsize=8)
    ax.text(*a_f, fraud_names[i] + " (A)", fontsize=9)
    ax.text(*a_r, real_names[i] + " (A)", fontsize=9)


ax.set_title("Embedding Space Before and After")

plt.show()


In [ ]:
before_df = pd.read_csv("embeddings_before.csv")
after_df  = pd.read_csv("embeddings_after.csv")

fraud_emb_cols = [c for c in before_df.columns if c.startswith("fraud_emb_")]
real_emb_cols  = [c for c in before_df.columns if c.startswith("real_emb_")]

fraud_before = before_df[fraud_emb_cols].to_numpy()
real_before  = before_df[real_emb_cols].to_numpy()

fraud_after  = after_df[fraud_emb_cols].to_numpy()
real_after   = after_df[real_emb_cols].to_numpy()

dist_before = np.linalg.norm(fraud_before - real_before, axis=1)
dist_after  = np.linalg.norm(fraud_after  - real_after,  axis=1)
change = dist_after - dist_before

mean_change = change.mean()

window = 0.05 * abs(mean_change)
candidate_idx = np.where(np.abs(change - mean_change) <= window)[0]

if len(candidate_idx) == 0:
    candidate_idx = np.arange(len(change))

sample_idx = np.random.choice(candidate_idx, size=3, replace=False)

all_embeddings = np.vstack([
    fraud_before, real_before,
    fraud_after,  real_after
])

pca = PCA(n_components=3, random_state=0)
all_pca = pca.fit_transform(all_embeddings)

N = len(before_df)

fraud_before_pca = all_pca[0:N]
real_before_pca  = all_pca[N:2*N]
fraud_after_pca  = all_pca[2*N:3*N]
real_after_pca   = all_pca[3*N:4*N]

fraud_names = before_df.loc[sample_idx, "fraudulent_name"].values
real_names  = before_df.loc[sample_idx, "real_name"].values
colors = plt.cm.tab10(np.linspace(0, 1, len(sample_idx)))

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

ax.set_xlim(-1, 1)
ax.set_ylim(-0.15, 0.15)
ax.set_zlim(-0.15, 0.15)

for i, color in enumerate(colors):
    b_f = fraud_before_pca[sample_idx[i]]
    b_r = real_before_pca[sample_idx[i]]
    a_f = fraud_after_pca[sample_idx[i]]
    a_r = real_after_pca[sample_idx[i]]

    ax.scatter(*b_f, color=color, s=60, alpha=0.3, marker="o")
    ax.scatter(*b_r, color=color, s=60, alpha=0.3, marker="o")

    ax.scatter(*a_f, color="orange", s=90, alpha=0.3, marker="o")
    ax.scatter(*a_r, color="orange", s=90, alpha=0.3, marker="o")

    ax.plot([b_f[0], b_r[0]], [b_f[1], b_r[1]], [b_f[2], b_r[2]],
            color=color, linestyle="--", alpha=0.6)
    ax.plot([a_f[0], a_r[0]], [a_f[1], a_r[1]], [a_f[2], a_r[2]],
            color="orange", linewidth=2)

    ax.text(*b_f, fraud_names[i] + " (B)", fontsize=8)
    ax.text(*b_r, real_names[i] + " (B)", fontsize=8)
    ax.text(*a_f, fraud_names[i] + " (A)", fontsize=9)
    ax.text(*a_r, real_names[i] + " (A)", fontsize=9)

ax.grid(False)
ax.set_title("PCA Embedding Space Before and After VA-TE")
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

before_df = pd.read_csv("embeddings_before.csv")
after_df  = pd.read_csv("embeddings_after.csv")

fraud_emb_cols = [c for c in before_df.columns if c.startswith("fraud_emb_")]
real_emb_cols  = [c for c in before_df.columns if c.startswith("real_emb_")]

fraud_before = before_df[fraud_emb_cols].to_numpy()
real_before  = before_df[real_emb_cols].to_numpy()

fraud_after  = after_df[fraud_emb_cols].to_numpy()
real_after   = after_df[real_emb_cols].to_numpy()


sample_idx = before_df.sample(n=2, random_state=42).index.to_numpy()

fraud_names = before_df.loc[sample_idx, "fraudulent_name"].values
real_names  = before_df.loc[sample_idx, "real_name"].values

fraud_before_s = fraud_before[sample_idx]
real_before_s  = real_before[sample_idx]

before_embeddings = np.vstack([fraud_before_s, real_before_s])
before_mean = before_embeddings.mean(axis=0)

pca = PCA(n_components=3, random_state=0)
before_pca = pca.fit_transform(before_embeddings)

N = len(sample_idx)

fraud_before_pca = before_pca[0:N]
real_before_pca  = before_pca[N:2*N]


fraud_after_s = fraud_after[sample_idx]
real_after_s  = real_after[sample_idx]

after_embeddings = np.vstack([fraud_after_s, real_after_s])
after_centered = after_embeddings - before_mean
after_pca = pca.transform(after_centered)

fraud_after_pca = after_pca[0:N]
real_after_pca  = after_pca[N:2*N]

colors = plt.cm.tab10(np.linspace(0, 1, len(sample_idx)))


fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

for i, color in enumerate(colors):
    b_f = fraud_before_pca[i]
    b_r = real_before_pca[i]
    a_f = fraud_after_pca[i]
    a_r = real_after_pca[i]

    ax.scatter(*b_f, color="blue", s=60, alpha=0.3, marker="o")
    ax.scatter(*b_r, color="blue", s=60, alpha=0.3, marker="o")

    ax.scatter(*a_f, color="orange", s=90, alpha=0.8, marker="o")
    ax.scatter(*a_r, color="orange", s=90, alpha=0.8, marker="o")

    ax.plot([b_f[0], b_r[0]], [b_f[1], b_r[1]], [b_f[2], b_r[2]],
            color="blue", linestyle="--", alpha=0.6)

    ax.plot([a_f[0], a_r[0]], [a_f[1], a_r[1]], [a_f[2], a_r[2]],
            color="orange", linewidth=2)

    ax.text(*b_f, fraud_names[i] + " (B)", fontsize=8)
    ax.text(*b_r, real_names[i] + " (B)", fontsize=8)
    ax.text(*a_f, fraud_names[i] + " (A)", fontsize=9)
    ax.text(*a_r, real_names[i] + " (A)", fontsize=9)

ax.set_title("Embedding Space Before and After VA-TE")
ax.grid(False)
plt.show()


# UMAP

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap
from mpl_toolkits.mplot3d import Axes3D

before_df = pd.read_csv("embeddings_before.csv")
after_df  = pd.read_csv("embeddings_after.csv")
before_df = before_df[before_df["label"]==1.0]
after_df  = after_df[after_df["label"]==1.0]

fraud_emb_cols = [c for c in before_df.columns if c.startswith("fraud_emb_")]
real_emb_cols  = [c for c in before_df.columns if c.startswith("real_emb_")]

fraud_before = before_df[fraud_emb_cols].to_numpy()
real_before  = before_df[real_emb_cols].to_numpy()

fraud_after  = after_df[fraud_emb_cols].to_numpy()
real_after   = after_df[real_emb_cols].to_numpy()


sample_idx = np.random.choice(len(before_df), size=3, replace=False)

fraud_names = before_df.iloc[sample_idx]["fraudulent_name"].values
real_names  = before_df.iloc[sample_idx]["real_name"].values

fraud_before_s = fraud_before[sample_idx]
real_before_s  = real_before[sample_idx]

before_embeddings = np.vstack([fraud_before_s, real_before_s])

umap_model = umap.UMAP(
    n_components=3,
    n_neighbors=5,
    min_dist=0.1,
    random_state=0
)

embedding = umap_model.fit_transform(before_embeddings)

N = len(sample_idx)

fraud_before_pca = embedding[0:N]
real_before_pca  = embedding[N:2*N]


fraud_after_s = fraud_after[sample_idx]
real_after_s  = real_after[sample_idx]

after_embeddings = np.vstack([fraud_after_s, real_after_s])
after_umap = umap_model.transform(after_embeddings)

fraud_after_pca = after_umap[0:N]
real_after_pca  = after_umap[N:2*N]


all_points = np.vstack([
    fraud_before_pca, real_before_pca,
    fraud_after_pca,  real_after_pca
])

mins = all_points.min(axis=0)
maxs = all_points.max(axis=0)

pad = 0.2 * (maxs - mins + 1e-12)
xlim = (mins[0] - pad[0], maxs[0] + pad[0])
ylim = (mins[1] - pad[1], maxs[1] + pad[1])
zlim = (mins[2] - pad[2], maxs[2] + pad[2])


fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

colors = plt.cm.tab10(np.linspace(0, 1, len(sample_idx)))
offset = 0.0002

for i, color in enumerate(colors):

    b_f = fraud_before_pca[i]
    b_r = real_before_pca[i]

    ax.scatter(*b_f, color="blue", s=90, alpha=0.3)
    ax.scatter(*b_r, color="blue", s=90, alpha=0.3)

    ax.plot(
        [b_f[0], b_r[0]],
        [b_f[1], b_r[1]],
        [b_f[2], b_r[2]],
        color="blue",
        linestyle="--",
        alpha=0.6
    )

    ax.text(b_f[0]+offset, b_f[1]+offset, b_f[2],
            fraud_names[i], fontsize=12)
    ax.text(b_r[0]+offset, b_r[1]+offset, b_r[2],
            real_names[i], fontsize=12)


    a_f = fraud_after_pca[i]
    a_r = real_after_pca[i]

    ax.scatter(*a_f, color="orange", s=90, alpha=0.5)
    ax.scatter(*a_r, color="orange", s=90, alpha=0.5)

    ax.plot(
        [a_f[0], a_r[0]],
        [a_f[1], a_r[1]],
        [a_f[2], a_r[2]],
        color="orange",
        linewidth=2
    )

    ax.text(a_f[0]+offset, a_f[1]+offset, a_f[2]+offset,
            fraud_names[i], fontsize=12)
    ax.text(a_r[0]+offset, a_r[1]+offset, a_r[2]+offset,
            real_names[i], fontsize=12)

ax.set_title("Embedding Space BEFORE (blue) and AFTER (orange) VA-TE")
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_zlim(zlim)
ax.grid(False)

plt.show()


# T-SNE

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openTSNE import TSNE
from mpl_toolkits.mplot3d import Axes3D 

before_df = pd.read_csv("embeddings_before.csv")
after_df  = pd.read_csv("embeddings_after.csv")
before_df = before_df[before_df["label"] == 1.0]
after_df  = after_df[after_df["label"] == 1.0]

fraud_emb_cols = [c for c in before_df.columns if c.startswith("fraud_emb_")]
real_emb_cols  = [c for c in before_df.columns if c.startswith("real_emb_")]

fraud_before = before_df[fraud_emb_cols].to_numpy()
real_before  = before_df[real_emb_cols].to_numpy()

fraud_after  = after_df[fraud_emb_cols].to_numpy()
real_after   = after_df[real_emb_cols].to_numpy()


sample_idx = np.random.choice(len(before_df), size=2, replace=False)

fraud_names = before_df.iloc[sample_idx]["fraudulent_name"].values
real_names  = before_df.iloc[sample_idx]["real_name"].values

fraud_before_s = fraud_before[sample_idx]
real_before_s  = real_before[sample_idx]
fraud_after_s  = fraud_after[sample_idx]
real_after_s   = real_after[sample_idx]


before_embeddings = np.vstack([fraud_before_s, real_before_s])

tsne = TSNE(
    n_components=3,
    perplexity=5,
    metric="euclidean",
    random_state=0,
    initialization="pca",
    n_jobs=1
)

before_tsne = tsne.fit(before_embeddings)

N = len(sample_idx)
fraud_before_tsne = np.asarray(before_tsne[0:N])
real_before_tsne  = np.asarray(before_tsne[N:2*N])


after_embeddings = np.vstack([fraud_after_s, real_after_s])
after_tsne = before_tsne.transform(after_embeddings)

fraud_after_tsne = np.asarray(after_tsne[0:N])
real_after_tsne  = np.asarray(after_tsne[N:2*N])


all_points = np.vstack([
    fraud_before_tsne, real_before_tsne,
    fraud_after_tsne,  real_after_tsne
])

mins = all_points.min(axis=0)
maxs = all_points.max(axis=0)

pad = 0.2 * (maxs - mins + 1e-12)
xlim = (mins[0] - pad[0], maxs[0] + pad[0])
ylim = (mins[1] - pad[1], maxs[1] + pad[1])
zlim = (mins[2] - pad[2], maxs[2] + pad[2])

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

colors = plt.cm.tab10(np.linspace(0, 1, len(sample_idx)))
offset = 0.0002

for i, color in enumerate(colors):
    b_f = fraud_before_tsne[i]
    b_r = real_before_tsne[i]

    ax.scatter(*b_f, color="blue", s=90, alpha=0.3)
    ax.scatter(*b_r, color="blue", s=90, alpha=0.3)

    ax.plot(
        [b_f[0], b_r[0]],
        [b_f[1], b_r[1]],
        [b_f[2], b_r[2]],
        color="blue",
        linestyle="--",
        alpha=0.6
    )

    ax.text(b_f[0]+offset, b_f[1]+offset, b_f[2],
            fraud_names[i], fontsize=12)
    ax.text(b_r[0]+offset, b_r[1]+offset, b_r[2],
            real_names[i], fontsize=12)

    a_f = fraud_after_tsne[i]
    a_r = real_after_tsne[i]

    ax.scatter(*a_f, color="orange", s=90, alpha=0.5)
    ax.scatter(*a_r, color="orange", s=90, alpha=0.5)

    ax.plot(
        [a_f[0], a_r[0]],
        [a_f[1], a_r[1]],
        [a_f[2], a_r[2]],
        color="orange",
        linewidth=2
    )

    ax.text(a_f[0]+offset, a_f[1]+offset, a_f[2]+offset,
            fraud_names[i], fontsize=12)
    ax.text(a_r[0]+offset, a_r[1]+offset, a_r[2]+offset,
            real_names[i], fontsize=12)

ax.set_title("Embedding Space BEFORE and AFTER VA-TE (t-SNE)")
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_zlim(zlim)
ax.grid(False)

plt.show()
